In [ ]:
## 41 CITY-WISE, Koppen-Geiger-ZONE-WISE, AND Temperature Homogeneous-ZONE-WISE CATEGORY HEATMAPS


In [1]:

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

# ============================================================
# 1. USER CONFIGURATION & PATHS
# ============================================================
CITY_OUTPUT_DIR = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_2")


SUMMARY_DIR_CITY = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_2\summary_long_wide")
ZONEWISE_SUMMARY_ROOT = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_2\summary")
KG_LOOKUP_CSV = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_2\KG_climate_city.csv")

# Output directory
MERGED_OUTDIR = CITY_OUTPUT_DIR / "output_1"
MERGED_OUTDIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RUN SWITCHES
# ------------------------------------------------------------
PLOT_YEARLY = True
PLOT_SEASONAL = True
SEASONS_TO_PLOT = ["MAM", "JJAS", "ON"]
CATEGORIES_TO_PLOT = ["Caution", "Extreme Caution", "Danger", "Extreme Danger"]

# ------------------------------------------------------------
# ORDERING & GROUPING
# ------------------------------------------------------------
CITY_ZONE_ORDER = ["Am", "Aw", "BSh", "As", "Cwa", "Csa", "BWh"]
CITY_ORDER_WITHIN_ZONE = "alphabetical"  # "alphabetical" or "input"
STRICT_ZONE_ORDER_ONLY = True  # Only plot zones listed in CITY_ZONE_ORDER

KG_ZONE_ORDER = ["Am", "Aw", "BSh", "As", "Cwa", "Csa", "BWh"]
TH_ZONE_ORDER = ["WC", "EC", "IP", "NC", "NW", "NE"]

ZONE_NAME_MAP = {
    "West Coast": "WC",
    "East Coast": "EC",
    "Interior Peninsula": "IP",
    "North Central": "NC",
    "North-West": "NW",
    "Northeast": "NE"
}

SOURCE_ORDER = [
    "historical",
    "ssp245_near",
    "ssp585_near",
    "ssp245_far",
    "ssp585_far",
]

FILE_LABELS = {
    "historical": "Historical",
    "ssp245_near": "SSP245 nf",
    "ssp585_near": "SSP585 nf",
    "ssp245_far": "SSP245 ff",
    "ssp585_far": "SSP585 ff",
}

SOURCE_ALIASES = {
    "historical": ["historical", "historical_imdaa", "imdaa"],
    "ssp245_near": ["ssp245_near", "ssp245_near_2026_2055", "ssp245nf"],
    "ssp585_near": ["ssp585_near", "ssp585_near_2026_2055", "ssp585nf"],
    "ssp245_far": ["ssp245_far", "ssp245_far_2070_2099", "ssp245ff"],
    "ssp585_far": ["ssp585_far", "ssp585_far_2070_2099", "ssp585ff"],
}

# ------------------------------------------------------------
# VISUAL & LAYOUT SETTINGS
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 300,  
    "savefig.dpi": 900,
})

# Overall combined figure size (Width, Height). 
FIGSIZE = (9.5, 12.0)

# Layout Spacing
PANEL_WSPACE = 0.05     
PANEL_HSPACE = 0.03    
TOP_MARGIN = 0.94
BOTTOM_MARGIN = 0.10
LEFT_MARGIN = 0.12
RIGHT_MARGIN = 0.91

# Heatmap appearance
HEATMAP_CMAP = "YlOrRd"
SHOW_PIXEL_LABELS = True
PIXEL_LABEL_FMT = "{:.0f}"
PIXEL_LABEL_FONT_SIZE = 8

# ============================================================
# PANEL LABEL CONFIGURATION
# ============================================================
DRAW_PANEL_LABELS = True
PANEL_LABEL_TOP = "(a)"
PANEL_LABEL_MIDDLE = "(b)"
PANEL_LABEL_BOTTOM = "(c)"
PANEL_LABEL_FONT_SIZE = 16
PANEL_LABEL_FONT_WEIGHT = "bold"

PANEL_LABEL_X = -0.5  
PANEL_LABEL_Y = 1.01  

# Top Panel Settings (City-wise)
DRAW_ZONE_SEPARATORS = True
ZONE_SEPARATOR_COLOR = "black"
ZONE_SEPARATOR_LINEWIDTH = 0.6
DRAW_ZONE_LABELS = True
ZONE_LINE_X = -3.2       
ZONE_TEXT_X = -3.5       
ZONE_ARROWSTYLE = "<|-|>"
ZONE_ARROW_SCALE = 10

# Colorbar Settings
COLORBAR_LABEL = "Average count"
COLORBAR_BBOX = [0.93, 0.15, 0.02, 0.70] 


# ============================================================
# 2. DATA LOADING HELPERS
# ============================================================
def _pretty_source(src):
    return FILE_LABELS.get(src, src)

def _normalize_source_key(name):
    low = str(name).lower()
    for key, aliases in SOURCE_ALIASES.items():
        for alias in aliases:
            if alias in low:
                return key
    for key in SOURCE_ORDER:
        if key in low:
            return key
    return str(name).lower()

# --- City Data Loaders ---
def _load_kg_lookup():
    if KG_LOOKUP_CSV is not None and Path(KG_LOOKUP_CSV).exists():
        df = pd.read_csv(KG_LOOKUP_CSV)
        df = df[["CityName", "ClimateZone"]].copy()
        df["CityName"] = df["CityName"].astype(str)
        df["ClimateZone"] = df["ClimateZone"].astype(str)
        return df.dropna().drop_duplicates("CityName")
    return pd.DataFrame(columns=["CityName", "ClimateZone"])

def _load_city_summary(mode, season=None):
    f = SUMMARY_DIR_CITY / f"summary_{mode}_categories_long.csv"
    if not f.exists():
        candidates = sorted(SUMMARY_DIR_CITY.glob(f"**/*{mode}*categories*long*.csv"))
        f = candidates[0] if candidates else None
    
    if f is None or not f.exists():
        return pd.DataFrame()

    df = pd.read_csv(f)
    for col in ["scenario_group", "season", "category", "city", "avg_30yr"]:
        if col not in df.columns: df[col] = np.nan

    df["scenario_group"] = df["scenario_group"].astype(str).str.lower()
    df["season"] = df["season"].fillna("").astype(str).str.upper()
    df["category"] = df["category"].astype(str)
    df["city"] = df["city"].astype(str)
    df["avg_30yr"] = pd.to_numeric(df["avg_30yr"], errors="coerce")

    df = df[df["scenario_group"].isin(SOURCE_ORDER)].copy()
    if season: df = df[df["season"] == season].copy()
    return df

# --- Zone Data Loaders ---
def _category_to_slug(category):
    return category.lower().replace(" ", "_")

def _find_zonewise_csv(zone_system, mode, category, season=None):
    root = ZONEWISE_SUMMARY_ROOT / zone_system / mode
    if not root.exists(): return None
    slug = _category_to_slug(category)
    candidates = []
    
    if mode == "yearly":
        pat = f"*{slug}*.csv"
        candidates = list(root.rglob(pat))
    else:
        if season is None: return None
        s = str(season).upper()
        pat = f"*{s}*{slug}*.csv"
        candidates = list(root.rglob(pat))
    return candidates[0] if candidates else None

def _load_zone_summary(zone_system, mode, season=None):
    records = []
    for cat in CATEGORIES_TO_PLOT:
        csv_path = _find_zonewise_csv(zone_system, mode, cat, season=season)
        if csv_path is None: continue
        
        df = pd.read_csv(csv_path)
        zone_col = next((c for c in df.columns if c.lower() in ["zone", "climatezone"]), df.columns[0])
        df = df.rename(columns={zone_col: "zone"})
        df["zone"] = df["zone"].astype(str).str.strip()

        # Apply Name Mapping for Temperature Homogeneous zones
        if zone_system == "temperature_homogeneous":
            df["zone"] = df["zone"].replace(ZONE_NAME_MAP)

        # Handle wide to long if necessary
        if "avg_30yr" not in df.columns:
            scenario_cols = [c for c in df.columns if c != "zone"]
            df = df.melt(id_vars=["zone"], value_vars=scenario_cols, var_name="scenario_group", value_name="avg_30yr")
        
        df["scenario_group"] = df["scenario_group"].map(lambda x: _normalize_source_key(x))
        df["avg_30yr"] = pd.to_numeric(df["avg_30yr"], errors="coerce")
        df = df[df["scenario_group"].isin(SOURCE_ORDER)].copy()
        df["category"] = cat
        records.append(df)
        
    if not records: return pd.DataFrame()
    return pd.concat(records, ignore_index=True)


# ============================================================
# 3. MATRIX BUILDERS & GROUPING
# ============================================================
def _ordered_cities_and_separators(cities):
    kg = _load_kg_lookup()
    cities = [str(c) for c in cities]
    if kg.empty: return sorted(cities), [], []

    kg = kg[kg["CityName"].isin(cities)].copy()
    ordered, separators, zone_ranges = [], [], []
    row_count = 0

    for zone in CITY_ZONE_ORDER:
        zdf = kg[kg["ClimateZone"] == zone].copy()
        if zdf.empty: continue
        if CITY_ORDER_WITHIN_ZONE == "alphabetical":
            zdf = zdf.sort_values("CityName", kind="stable")
        
        zone_cities = [c for c in zdf["CityName"] if c in cities and c not in ordered]
        if not zone_cities: continue
        
        start_row = row_count
        ordered.extend(zone_cities)
        row_count += len(zone_cities)
        end_row = row_count - 1
        
        zone_ranges.append((zone, start_row, end_row))
        separators.append(row_count - 0.5)

    remaining = [c for c in cities if c not in ordered]
    if remaining:
        if CITY_ORDER_WITHIN_ZONE == "alphabetical": remaining = sorted(remaining)
        ordered.extend(remaining)
    if separators and separators[-1] >= len(ordered)-0.5:
        separators = separators[:-1]
        
    return ordered, separators, zone_ranges

def _make_city_matrix(df, category):
    q = df[df["category"].astype(str).str.lower() == category.lower()].copy()
    if q.empty: return pd.DataFrame(), [], []
    mat = q.pivot_table(index="city", columns="scenario_group", values="avg_30yr", aggfunc="mean")
    ordered_cities, separators, zone_ranges = _ordered_cities_and_separators(mat.index.tolist())
    
    mat = mat.reindex(index=ordered_cities)
    scens = [s for s in SOURCE_ORDER if s in mat.columns]
    mat = mat.reindex(columns=scens)
    return mat, separators, zone_ranges

def _make_zone_matrix(df, category, zone_system):
    q = df[df["category"].astype(str).str.lower() == category.lower()].copy()
    if q.empty: return pd.DataFrame()
    mat = q.pivot_table(index="zone", columns="scenario_group", values="avg_30yr", aggfunc="mean")
    
    zones = mat.index.tolist()
    
    # Select ordering based on zone_system
    order_list = KG_ZONE_ORDER if zone_system == "koppen_geiger" else TH_ZONE_ORDER
    ordered = [z for z in order_list if z in zones]
    
    if not STRICT_ZONE_ORDER_ONLY:
        extras = [z for z in zones if z not in ordered]
        ordered += extras
        
    mat = mat.reindex(index=ordered)
    scens = [s for s in SOURCE_ORDER if s in mat.columns]
    mat = mat.reindex(columns=scens)
    return mat

# ============================================================
# 4. PLOTTING LOGIC
# ============================================================
def _add_pixel_labels(ax, mat):
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.iat[i, j]
            if pd.notna(v):
                ax.text(j, i, PIXEL_LABEL_FMT.format(v),
                        ha="center", va="center",
                        fontsize=PIXEL_LABEL_FONT_SIZE,
                        color="black")

def _add_zone_labels(ax, zone_ranges):
    if not DRAW_ZONE_LABELS: return
    x = ZONE_LINE_X
    for zone, start_row, end_row in zone_ranges:
        y0 = start_row - 0.30
        y1 = end_row + 0.30
        arr = FancyArrowPatch((x, y0), (x, y1), arrowstyle=ZONE_ARROWSTYLE,
                              mutation_scale=ZONE_ARROW_SCALE, linewidth=1.0,
                              color="black", clip_on=False, zorder=100)
        ax.add_patch(arr)
        ax.text(ZONE_TEXT_X, (y0+y1)/2, zone, rotation=90,
                ha="center", va="center", fontsize=12,
                fontweight="bold", color="black", clip_on=False, zorder=100)

def _plot_merged_combined(city_df, kg_df, th_df, mode, season=None):
    if city_df.empty or kg_df.empty or th_df.empty: return
    
    city_mats, kg_mats, th_mats, titles = [], [], [], []
    zone_seps, zone_ranges = None, None
    
    for cat in CATEGORIES_TO_PLOT:
        c_mat, seps, zr = _make_city_matrix(city_df, cat)
        kg_mat = _make_zone_matrix(kg_df, cat, "koppen_geiger")
        th_mat = _make_zone_matrix(th_df, cat, "temperature_homogeneous")
        
        if not c_mat.empty and not kg_mat.empty and not th_mat.empty:
            city_mats.append(c_mat)
            kg_mats.append(kg_mat)
            th_mats.append(th_mat)
            titles.append(cat)
            if zone_seps is None:
                zone_seps = seps
                zone_ranges = zr
                
    if not city_mats: return
    
    # Calculate global scale
    all_vals = np.concatenate(
        [m.values.ravel() for m in city_mats] + 
        [m.values.ravel() for m in kg_mats] + 
        [m.values.ravel() for m in th_mats]
    )
    all_vals = all_vals[np.isfinite(all_vals)]
    vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
    
    # Calculate dynamic height ratio based on row count
    city_rows = city_mats[0].shape[0]
    kg_rows = kg_mats[0].shape[0]
    th_rows = th_mats[0].shape[0]
    
    fig = plt.figure(figsize=FIGSIZE)
    # 3 Rows (City, KG Zone, TH Zone), 4 Columns (Categories)
    gs = gridspec.GridSpec(3, len(city_mats), 
                           height_ratios=[city_rows, kg_rows, th_rows], 
                           hspace=PANEL_HSPACE, wspace=PANEL_WSPACE)
    
    im = None
    for c, cat in enumerate(titles):
        c_mat = city_mats[c]
        k_mat = kg_mats[c]
        t_mat = th_mats[c]
        
        # ----------------------------------------------------
        # TOP ROW: City Data
        # ----------------------------------------------------
        ax_top = fig.add_subplot(gs[0, c])
        im = ax_top.imshow(c_mat.values, aspect='auto', interpolation='nearest', cmap=HEATMAP_CMAP, vmin=vmin, vmax=vmax)
        
        ax_top.set_title(cat, fontsize=14, fontweight="bold", pad=5)
        # Hide X-ticks completely to share with bottom rows
        ax_top.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
        
        ax_top.set_yticks(np.arange(c_mat.shape[0]))
        if c == 0:
            ax_top.set_yticklabels(c_mat.index.tolist(), fontsize=10, fontweight='bold')
            _add_zone_labels(ax_top, zone_ranges)
            
            if DRAW_PANEL_LABELS:
                ax_top.text(PANEL_LABEL_X, PANEL_LABEL_Y, PANEL_LABEL_TOP,
                            transform=ax_top.transAxes, fontsize=PANEL_LABEL_FONT_SIZE,
                            fontweight=PANEL_LABEL_FONT_WEIGHT, ha='center', va='bottom', clip_on=False)
        else:
            ax_top.tick_params(labelleft=False)
            
        if DRAW_ZONE_SEPARATORS and zone_seps:
            for y in zone_seps:
                ax_top.axhline(y=y, color=ZONE_SEPARATOR_COLOR, linewidth=ZONE_SEPARATOR_LINEWIDTH)
        if SHOW_PIXEL_LABELS: _add_pixel_labels(ax_top, c_mat)

        # ----------------------------------------------------
        # MIDDLE ROW: Koppen-Geiger Zone Data
        # ----------------------------------------------------
        ax_mid = fig.add_subplot(gs[1, c])
        ax_mid.imshow(k_mat.values, aspect='auto', interpolation='nearest', cmap=HEATMAP_CMAP, vmin=vmin, vmax=vmax)
        
        # Hide X-ticks completely
        ax_mid.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
        
        ax_mid.set_yticks(np.arange(k_mat.shape[0]))
        if c == 0:
            ax_mid.set_yticklabels(k_mat.index.tolist(), fontsize=10, fontweight='bold')
            
            if DRAW_PANEL_LABELS:
                ax_mid.text(-0.5, 0.85, PANEL_LABEL_MIDDLE,
                            transform=ax_mid.transAxes, fontsize=PANEL_LABEL_FONT_SIZE,
                            fontweight=PANEL_LABEL_FONT_WEIGHT, ha='center', va='bottom', clip_on=False)
        else:
            ax_mid.tick_params(labelleft=False)
            
        if SHOW_PIXEL_LABELS: _add_pixel_labels(ax_mid, k_mat)
            
        # ----------------------------------------------------
        # BOTTOM ROW: Temperature Homogeneous Zone Data
        # ----------------------------------------------------
        ax_bot = fig.add_subplot(gs[2, c])
        ax_bot.imshow(t_mat.values, aspect='auto', interpolation='nearest', cmap=HEATMAP_CMAP, vmin=vmin, vmax=vmax)
        
        # Show X-ticks on the very bottom row only
        ax_bot.set_xticks(np.arange(t_mat.shape[1]))
        ax_bot.set_xticklabels([_pretty_source(col) for col in t_mat.columns], 
                               rotation=45, ha="right", fontsize=9, fontweight="bold")
        
        ax_bot.set_yticks(np.arange(t_mat.shape[0]))
        if c == 0:
            ax_bot.set_yticklabels(t_mat.index.tolist(), fontsize=10, fontweight='bold')
            
            if DRAW_PANEL_LABELS:
                ax_bot.text(-0.5, 0.85, PANEL_LABEL_BOTTOM,
                            transform=ax_bot.transAxes, fontsize=PANEL_LABEL_FONT_SIZE,
                            fontweight=PANEL_LABEL_FONT_WEIGHT, ha='center', va='bottom', clip_on=False)
        else:
            ax_bot.tick_params(labelleft=False)
            
        if SHOW_PIXEL_LABELS: _add_pixel_labels(ax_bot, t_mat)

    # Adjust Layout and Add Colorbar
    fig.subplots_adjust(left=LEFT_MARGIN, right=RIGHT_MARGIN, top=TOP_MARGIN, bottom=BOTTOM_MARGIN)
    cax = fig.add_axes(COLORBAR_BBOX)
    cbar = fig.colorbar(im, cax=cax, orientation="vertical")
    cbar.set_label(COLORBAR_LABEL, fontsize=14, fontweight="bold")
    cbar.ax.tick_params(labelsize=12)

    # Save Outputs
    season_txt = f"_{season}" if season else ""
    base_name = MERGED_OUTDIR / f"merged_city_kg_th_{mode}{season_txt}"
    
    formats = ['png', 'svg']
    for fmt in formats:
        out_path = base_name.with_suffix(f'.{fmt}')
        plt.savefig(out_path, format=fmt, dpi=600, bbox_inches="tight")
        print(f"Saved: {out_path.name}")
        
    plt.close(fig)

# ============================================================
# 5. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    print(f"Generating merged heatmaps in: {MERGED_OUTDIR}\n")
    
    if PLOT_YEARLY:
        print("Processing Yearly Data...")
        cdf = _load_city_summary("yearly")
        kg_df = _load_zone_summary("koppen_geiger", "yearly")
        th_df = _load_zone_summary("temperature_homogeneous", "yearly")
        
        if not cdf.empty and not kg_df.empty and not th_df.empty:
            _plot_merged_combined(cdf, kg_df, th_df, mode="yearly")
        else:
            print("  Missing yearly data for cities, KG zones, or TH zones.")

    if PLOT_SEASONAL:
        print("\nProcessing Seasonal Data...")
        cdf = _load_city_summary("seasonal")
        for season in SEASONS_TO_PLOT:
            print(f"  {season}...")
            c_season = cdf[cdf["season"] == season] if not cdf.empty else pd.DataFrame()
            kg_season = _load_zone_summary("koppen_geiger", "seasonal", season=season)
            th_season = _load_zone_summary("temperature_homogeneous", "seasonal", season=season)
            
            if not c_season.empty and not kg_season.empty and not th_season.empty:
                _plot_merged_combined(c_season, kg_season, th_season, mode="seasonal", season=season)
            else:
                print(f"    Missing seasonal data for {season}.")
                
    print("\n✓ Merged heatmaps complete.")

Generating merged heatmaps in: C:\Users\PC1\Desktop\Code_Data\Data\Figure_2\output_1

Processing Yearly Data...
Saved: merged_city_kg_th_yearly.png
Saved: merged_city_kg_th_yearly.svg

Processing Seasonal Data...
  MAM...
Saved: merged_city_kg_th_seasonal_MAM.png
Saved: merged_city_kg_th_seasonal_MAM.svg
  JJAS...
Saved: merged_city_kg_th_seasonal_JJAS.png
Saved: merged_city_kg_th_seasonal_JJAS.svg
  ON...
Saved: merged_city_kg_th_seasonal_ON.png
Saved: merged_city_kg_th_seasonal_ON.svg

✓ Merged heatmaps complete.


In [ ]:
## ZONE-WISE CATEGORY HEATMAPS 
## 4 Temporal Scales (YEARLY, MAM, JJAS, ON) x 4 Categories

In [3]:

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ============================================================
# 1. USER CONFIGURATION & PATHS
# ============================================================
# Base output directory
CITY_OUTPUT_DIR = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_2")
ZONEWISE_SUMMARY_ROOT = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_2\summary")

# Output directory for the new specific zone plots
MERGED_OUTDIR = CITY_OUTPUT_DIR / "output_2"
MERGED_OUTDIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# RUN SWITCHES
# ------------------------------------------------------------
ZONE_SYSTEMS = ["koppen_geiger", "temperature_homogeneous"]
PERIODS_TO_PLOT = ["YEARLY", "MAM", "JJAS", "ON"]
CATEGORIES_TO_PLOT = ["Caution", "Extreme Caution", "Danger", "Extreme Danger"]

# ------------------------------------------------------------
# ORDERING & GROUPING
# ------------------------------------------------------------
ZONE_ORDERS = {
    "temperature_homogeneous": [
        "WC", "EC", "IP", 
        "NC", "NW", "NE"
    ],
    "koppen_geiger": [
        "Am", "Aw", "BSh", "As", "Cwa", "Csa", "BWh"
    ]
}

# Mapping dictionary to convert long names in input files to abbreviated labels
ZONE_NAME_MAP = {
    "West Coast": "WC",
    "East Coast": "EC",
    "Interior Peninsula": "IP",
    "North Central": "NC",
    "North-West": "NW",
    "Northeast": "NE"
}

STRICT_ZONE_ORDER_ONLY = True  

SOURCE_ORDER = [
    "historical",
    "ssp245_near",
    "ssp585_near",
    "ssp245_far",
    "ssp585_far",
]

FILE_LABELS = {
    "historical": "Historical",
    "ssp245_near": "SSP245 nf",
    "ssp585_near": "SSP585 nf",
    "ssp245_far": "SSP245 ff",
    "ssp585_far": "SSP585 ff",
}

SOURCE_ALIASES = {
    "historical": ["historical", "historical_imdaa", "imdaa"],
    "ssp245_near": ["ssp245_near", "ssp245_near_2026_2055", "ssp245nf"],
    "ssp585_near": ["ssp585_near", "ssp585_near_2026_2055", "ssp585nf"],
    "ssp245_far": ["ssp245_far", "ssp245_far_2070_2099", "ssp245ff"],
    "ssp585_far": ["ssp585_far", "ssp585_far_2070_2099", "ssp585ff"],
}

# ------------------------------------------------------------
# VISUAL & LAYOUT SETTINGS
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 1200,
    "savefig.dpi": 1200,
})


FIGSIZE_MAP = {
    "koppen_geiger": (8.5, 8.0),            
    "temperature_homogeneous": (8.5, 7.0)    
}

PANEL_WSPACE = 0.05    
PANEL_HSPACE = 0.05   
TOP_MARGIN = 0.94
BOTTOM_MARGIN = 0.10
LEFT_MARGIN = 0.15
RIGHT_MARGIN = 0.90

HEATMAP_CMAP = "YlOrRd"
SHOW_PIXEL_LABELS = True
PIXEL_LABEL_FMT = "{:.0f}"
PIXEL_LABEL_FONT_SIZE = 8

PANEL_LABELS = {"YEARLY": "(a) YEARLY", "MAM": "(b) MAM", "JJAS": "(c) JJAS", "ON": "(d) ON"}
PANEL_LABEL_FONT_SIZE = 14
PANEL_LABEL_FONT_WEIGHT = "bold"

COLORBAR_LABEL = "Average count"
COLORBAR_BBOX = [0.92, 0.15, 0.02, 0.70]


# ============================================================
# 2. DATA LOADING HELPERS
# ============================================================
def _pretty_source(src):
    return FILE_LABELS.get(src, src)

def _normalize_source_key(name):
    low = str(name).lower()
    for key, aliases in SOURCE_ALIASES.items():
        for alias in aliases:
            if alias in low:
                return key
    for key in SOURCE_ORDER:
        if key in low:
            return key
    return str(name).lower()

def _category_to_slug(category):
    return category.lower().replace(" ", "_")

def _find_zonewise_csv(zone_system, mode, category, season=None):
    root = ZONEWISE_SUMMARY_ROOT / zone_system / mode
    if not root.exists(): return None
    slug = _category_to_slug(category)
    
    if mode == "yearly":
        pat = f"*{slug}*.csv"
        candidates = list(root.rglob(pat))
    else:
        if season is None: return None
        s = str(season).upper()
        pat = f"*{s}*{slug}*.csv"
        candidates = list(root.rglob(pat))
    return candidates[0] if candidates else None

def _load_zone_summary(zone_system, mode, season=None):
    records = []
    for cat in CATEGORIES_TO_PLOT:
        csv_path = _find_zonewise_csv(zone_system, mode, cat, season=season)
        if csv_path is None: continue
        
        df = pd.read_csv(csv_path)
        zone_col = next((c for c in df.columns if c.lower() in ["zone", "climatezone"]), df.columns[0])
        df = df.rename(columns={zone_col: "zone"})
        
        # Apply the mapping for Abbreviations here
        df["zone"] = df["zone"].astype(str).str.strip()
        df["zone"] = df["zone"].replace(ZONE_NAME_MAP)

        if "avg_30yr" not in df.columns:
            scenario_cols = [c for c in df.columns if c != "zone"]
            df = df.melt(id_vars=["zone"], value_vars=scenario_cols, var_name="scenario_group", value_name="avg_30yr")
        
        df["scenario_group"] = df["scenario_group"].map(lambda x: _normalize_source_key(x))
        df["avg_30yr"] = pd.to_numeric(df["avg_30yr"], errors="coerce")
        df = df[df["scenario_group"].isin(SOURCE_ORDER)].copy()
        df["category"] = cat
        records.append(df)
        
    if not records: return pd.DataFrame()
    return pd.concat(records, ignore_index=True)

# ============================================================
# 3. MATRIX BUILDER
# ============================================================
def _make_zone_matrix(df, category, zone_system):
    q = df[df["category"].astype(str).str.lower() == category.lower()].copy()
    if q.empty: return pd.DataFrame()
    mat = q.pivot_table(index="zone", columns="scenario_group", values="avg_30yr", aggfunc="mean")
    
    zones = mat.index.tolist()
    ordered = [z for z in ZONE_ORDERS[zone_system] if z in zones]
    if not STRICT_ZONE_ORDER_ONLY:
        extras = [z for z in zones if z not in ordered]
        ordered += extras
        
    mat = mat.reindex(index=ordered)
    scens = [s for s in SOURCE_ORDER if s in mat.columns]
    mat = mat.reindex(columns=scens)
    return mat

# ============================================================
# 4. PLOTTING LOGIC
# ============================================================
def _add_pixel_labels(ax, mat):
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.iat[i, j]
            if pd.notna(v):
                ax.text(j, i, PIXEL_LABEL_FMT.format(v),
                        ha="center", va="center",
                        fontsize=PIXEL_LABEL_FONT_SIZE,
                        color="black")

def generate_figure_for_zone_system(zone_system):
    print(f"\nProcessing Zone System: {zone_system}")
    
    # 1. Pre-load all data to determine global colorbar scale
    matrix_dict = {}
    all_vals = []
    
    for period in PERIODS_TO_PLOT:
        mode = "yearly" if period == "YEARLY" else "seasonal"
        season = None if period == "YEARLY" else period
        
        df = _load_zone_summary(zone_system, mode, season=season)
        if df.empty:
            print(f"  Missing data for {period}")
            continue
            
        matrix_dict[period] = {}
        for cat in CATEGORIES_TO_PLOT:
            mat = _make_zone_matrix(df, cat, zone_system)
            if not mat.empty:
                matrix_dict[period][cat] = mat
                all_vals.extend(mat.values.ravel())
    
    if not all_vals:
        print("  No data found to plot.")
        return

    all_vals = np.array(all_vals)
    all_vals = all_vals[np.isfinite(all_vals)]
    vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
    
    # 2. Setup Figure
    figsize = FIGSIZE_MAP.get(zone_system, (11, 10))
    fig = plt.figure(figsize=figsize)
    
    # 4 Rows (Temporal periods), 4 Columns (Categories)
    gs = gridspec.GridSpec(len(PERIODS_TO_PLOT), len(CATEGORIES_TO_PLOT), 
                           hspace=PANEL_HSPACE, wspace=PANEL_WSPACE)
    
    im = None
    for r, period in enumerate(PERIODS_TO_PLOT):
        if period not in matrix_dict: continue
        
        for c, cat in enumerate(CATEGORIES_TO_PLOT):
            if cat not in matrix_dict[period]: continue
                
            mat = matrix_dict[period][cat]
            ax = fig.add_subplot(gs[r, c])
            
            im = ax.imshow(mat.values, aspect='auto', interpolation='nearest', 
                           cmap=HEATMAP_CMAP, vmin=vmin, vmax=vmax)
            
            # --- Titles (Only on Top Row) ---
            if r == 0:
                ax.set_title(cat, fontsize=14, fontweight="bold", pad=8)
                
            # --- Y-Axis (Only on First Column) ---
            ax.set_yticks(np.arange(mat.shape[0]))
            if c == 0:
                ax.set_yticklabels(mat.index.tolist(), fontsize=11, fontweight='bold')
                # Add Panel Label outside left edge
                ax.text(-0.8, 0.95, PANEL_LABELS[period],                                    
                        transform=ax.transAxes, fontsize=PANEL_LABEL_FONT_SIZE,
                        fontweight=PANEL_LABEL_FONT_WEIGHT, ha='left', va='bottom', clip_on=False)
            else:
                ax.tick_params(labelleft=False)
                
            # --- X-Axis (Only on Bottom Row) ---
            ax.set_xticks(np.arange(mat.shape[1]))
            if r == len(PERIODS_TO_PLOT) - 1:
                ax.set_xticklabels([_pretty_source(col) for col in mat.columns], 
                                   rotation=45, ha="right", fontsize=10, fontweight="bold")
            else:
                ax.tick_params(labelbottom=False, bottom=False)
                
            if SHOW_PIXEL_LABELS: _add_pixel_labels(ax, mat)

    # 3. Add Shared Colorbar
    fig.subplots_adjust(left=LEFT_MARGIN, right=RIGHT_MARGIN, top=TOP_MARGIN, bottom=BOTTOM_MARGIN)
    cax = fig.add_axes(COLORBAR_BBOX)
    cbar = fig.colorbar(im, cax=cax, orientation="vertical")
    cbar.set_label(COLORBAR_LABEL, fontsize=14, fontweight="bold")
    cbar.ax.tick_params(labelsize=12)

    # 4. Save Outputs
    output_formats = ['svg', 'png']
    base_out_name = f"Figure_S9_Style_{zone_system}"
    
    for fmt in output_formats:
        out_path = MERGED_OUTDIR / f"{base_out_name}.{fmt}"
        plt.savefig(out_path, format=fmt, dpi=1200, bbox_inches="tight")
        print(f"Saved: {out_path.name}")
        
    plt.close(fig)

# ============================================================
# 5. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    print(f"Generating combined heatmaps in: {MERGED_OUTDIR}\n")
    for zone in ZONE_SYSTEMS:
        generate_figure_for_zone_system(zone)
    print("\n✓ Processing complete.")

Generating combined heatmaps in: C:\Users\PC1\Desktop\Code_Data\Data\Figure_2\output_2


Processing Zone System: koppen_geiger
Saved: Figure_S9_Style_koppen_geiger.svg
Saved: Figure_S9_Style_koppen_geiger.png

Processing Zone System: temperature_homogeneous
Saved: Figure_S9_Style_temperature_homogeneous.svg
Saved: Figure_S9_Style_temperature_homogeneous.png

✓ Processing complete.
